# Module 3: Convolutional Neural Networks

This notebook covers CNN fundamentals with hands-on implementations.

**Topics covered:**
- Convolution operation
- Pooling layers
- Building CNNs from scratch
- Using PyTorch for CNNs

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 3.1 The Convolution Operation

In [ ]:
def conv2d_naive(image, kernel, stride=1, padding=0):
    """
    Naive 2D convolution implementation.
    
    Args:
        image: Input image (H, W)
        kernel: Filter (kH, kW)
        stride: Step size
        padding: Zero padding
    
    Returns:
        Output feature map
    """
    # Add padding
    if padding > 0:
        image = np.pad(image, padding, mode='constant', constant_values=0)
    
    H, W = image.shape
    kH, kW = kernel.shape
    
    # Output dimensions
    out_H = (H - kH) // stride + 1
    out_W = (W - kW) // stride + 1
    
    output = np.zeros((out_H, out_W))
    
    for i in range(out_H):
        for j in range(out_W):
            # Extract patch
            patch = image[i*stride:i*stride+kH, j*stride:j*stride+kW]
            # Element-wise multiply and sum
            output[i, j] = np.sum(patch * kernel)
    
    return output

In [ ]:
# Define common edge detection kernels
kernels = {
    'horizontal_edge': np.array([[-1, -1, -1],
                                  [0,  0,  0],
                                  [1,  1,  1]]),
    'vertical_edge': np.array([[-1, 0, 1],
                                [-1, 0, 1],
                                [-1, 0, 1]]),
    'sobel_x': np.array([[-1, 0, 1],
                         [-2, 0, 2],
                         [-1, 0, 1]]),
    'sobel_y': np.array([[-1, -2, -1],
                         [0,  0,  0],
                         [1,  2,  1]]),
    'sharpen': np.array([[0, -1,  0],
                         [-1, 5, -1],
                         [0, -1,  0]]),
    'blur': np.ones((3, 3)) / 9
}

In [ ]:
# Create a simple test image with edges
test_image = np.zeros((32, 32))
test_image[8:24, 8:24] = 1  # White square
test_image[12:20, 12:20] = 0  # Black square inside

# Apply different kernels
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

axes[0, 0].imshow(test_image, cmap='gray')
axes[0, 0].set_title('Original')
axes[0, 0].axis('off')

for idx, (name, kernel) in enumerate(kernels.items()):
    row, col = (idx + 1) // 4, (idx + 1) % 4
    output = conv2d_naive(test_image, kernel, padding=1)
    axes[row, col].imshow(output, cmap='gray')
    axes[row, col].set_title(name)
    axes[row, col].axis('off')

axes[1, 3].axis('off')
plt.tight_layout()
plt.suptitle('Convolution with Different Kernels', fontsize=14, y=1.02)
plt.show()

## 3.2 Pooling Layers

In [ ]:
def max_pool2d(x, pool_size=2, stride=2):
    """Max pooling operation."""
    H, W = x.shape
    out_H = (H - pool_size) // stride + 1
    out_W = (W - pool_size) // stride + 1
    
    output = np.zeros((out_H, out_W))
    
    for i in range(out_H):
        for j in range(out_W):
            patch = x[i*stride:i*stride+pool_size, j*stride:j*stride+pool_size]
            output[i, j] = np.max(patch)
    
    return output

def avg_pool2d(x, pool_size=2, stride=2):
    """Average pooling operation."""
    H, W = x.shape
    out_H = (H - pool_size) // stride + 1
    out_W = (W - pool_size) // stride + 1
    
    output = np.zeros((out_H, out_W))
    
    for i in range(out_H):
        for j in range(out_W):
            patch = x[i*stride:i*stride+pool_size, j*stride:j*stride+pool_size]
            output[i, j] = np.mean(patch)
    
    return output

In [ ]:
# Demonstrate pooling
feature_map = np.random.rand(8, 8)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(feature_map, cmap='viridis')
axes[0].set_title(f'Original (8x8)')

max_pooled = max_pool2d(feature_map)
axes[1].imshow(max_pooled, cmap='viridis')
axes[1].set_title(f'Max Pool 2x2 ({max_pooled.shape[0]}x{max_pooled.shape[1]})')

avg_pooled = avg_pool2d(feature_map)
axes[2].imshow(avg_pooled, cmap='viridis')
axes[2].set_title(f'Avg Pool 2x2 ({avg_pooled.shape[0]}x{avg_pooled.shape[1]})')

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3.3 Receptive Field Calculation

In [ ]:
def calculate_receptive_field(layers):
    """
    Calculate receptive field for a sequence of conv/pool layers.
    
    Each layer is a tuple: (kernel_size, stride)
    """
    rf = 1  # Initial receptive field
    total_stride = 1
    
    print("Layer | Kernel | Stride | RF | Total Stride")
    print("-" * 50)
    
    for i, (k, s) in enumerate(layers):
        rf = rf + (k - 1) * total_stride
        total_stride *= s
        print(f"{i+1:5} | {k:6} | {s:6} | {rf:2} | {total_stride}")
    
    return rf

# Example: VGG-style network
print("VGG-style layers:")
vgg_layers = [
    (3, 1),  # Conv 3x3
    (3, 1),  # Conv 3x3
    (2, 2),  # MaxPool 2x2
    (3, 1),  # Conv 3x3
    (3, 1),  # Conv 3x3
    (2, 2),  # MaxPool 2x2
]
rf = calculate_receptive_field(vgg_layers)
print(f"\nFinal receptive field: {rf}x{rf}")

## 3.4 CNN with PyTorch

In [ ]:
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torchvision import datasets, transforms
    TORCH_AVAILABLE = True
except ImportError:
    print("PyTorch not available. Install with: pip install torch torchvision")
    TORCH_AVAILABLE = False

In [ ]:
if TORCH_AVAILABLE:
    class SimpleCNN(nn.Module):
        """A simple CNN for MNIST."""
        def __init__(self):
            super().__init__()
            # Conv layers
            self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
            self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
            self.pool = nn.MaxPool2d(2, 2)
            
            # FC layers
            self.fc1 = nn.Linear(64 * 7 * 7, 128)
            self.fc2 = nn.Linear(128, 10)
            self.dropout = nn.Dropout(0.25)
        
        def forward(self, x):
            # Conv block 1
            x = self.pool(F.relu(self.conv1(x)))  # 28x28 -> 14x14
            # Conv block 2
            x = self.pool(F.relu(self.conv2(x)))  # 14x14 -> 7x7
            # Flatten
            x = x.view(-1, 64 * 7 * 7)
            # FC layers
            x = F.relu(self.fc1(x))
            x = self.dropout(x)
            x = self.fc2(x)
            return x
    
    # Create model and print summary
    model = SimpleCNN()
    print(model)
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\nTotal parameters: {total_params:,}")

## 3.5 Visualizing CNN Filters

In [ ]:
if TORCH_AVAILABLE:
    # Get first layer filters
    filters = model.conv1.weight.data.numpy()
    print(f"First conv layer filters shape: {filters.shape}")
    
    # Plot first 16 filters
    fig, axes = plt.subplots(4, 8, figsize=(12, 6))
    for i, ax in enumerate(axes.flat):
        if i < filters.shape[0]:
            ax.imshow(filters[i, 0], cmap='gray')
        ax.axis('off')
    plt.suptitle('First Conv Layer Filters (randomly initialized)', fontsize=12)
    plt.tight_layout()
    plt.show()

## Summary

In this notebook, we:
1. Implemented 2D convolution from scratch
2. Applied edge detection kernels
3. Built max and average pooling
4. Calculated receptive fields
5. Created a CNN with PyTorch

**Next:** Module 4 covers Sequence Models (RNNs, LSTMs).